# 06 - calving fronts and mean distance error (MDE)

**GPU runtime, high RAM.** Arm 1 fronts, scored on the validation glacier
SI (test glaciers stay untouched until 07).

**Route (decided 2026-09-24): fronts from zone maps, as the official CaFFe
zone model does.** Every model predicts **4 classes** -- NA, rock, glacier,
ocean -- so it never draws fronts inside radar shadow. Then, exactly as the
official code (verified identical on 25 random cases):

1. fill ocean gaps, keep the largest ocean component
2. front = ocean pixels touching glacier; drop pieces <= 750 m long
3. zero everything outside the image's bounding box
4. MDE = mean of **all** front-pixel distances (both directions) in metres,
   pooled over images; images with no predicted front are counted, not scored

Frozen encoders: 4-class linear probe per 16 px patch, logits bilinearly
upsampled to pixels. The 16 px grid limits front precision to a few pixels --
the gap AnyUp targets (tried in 08). U-Net: dense per-pixel prediction.

For later (07, **test** glaciers, not comparable with SI): CaFFe baseline
all 122 images 753 +/- 76 m, Mapple 150 +/- 24 m, Columbia 840 +/- 84 m;
AMD-HookNet 438 m; HookFormer 353 +/- 16 m; AMD-HookNet++ 367 m.

In [1]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# --- cell: secrets, GPU, data ---
import os, shutil
from pathlib import Path
import torch
from sartransfer.env import load_env, require
from sartransfer.data import inventory as inv

load_env()
require("HF_TOKEN", "gated model weights")
assert torch.cuda.is_available(), "switch to a GPU runtime"
print("gpu:", torch.cuda.get_device_properties(0).name)
DRIVE = Path("/content/drive/MyDrive")
RAW, RES = DRIVE / "sar-transfer" / "raw", DRIVE / "sar-transfer" / "results"
CAFFE, FEAT = Path("/content/caffe"), Path("/content/features")
inv.unzip_caffe(inv.fetch_caffe_zip(RAW), CAFFE)
DATA = inv.find_data_root(CAFFE)
df = inv.scan(DATA, subdirs=("sar_images",))
print(len(df), "SAR images | free disk:", f"{shutil.disk_usage('/content').free/1e9:.0f} GB")

In [4]:
# --- cell: FROZEN ENCODERS -> 4-class probe -> fronts -> MDE (one loop) ---
import time
from sartransfer.runs import run_fronts

RUNS = [                                    # caffe for every encoder, SARATR-X unit (see README, Encoders)
    ("cradio-v4-h", "caffe"),
    ("dinov3-l-photo", "caffe"),
    ("dinov3-l-sat", "caffe"),
    ("saratrx-v1", "unit"),
    ("terramind-v1-base-s1", "caffe"),
]
t0, status = time.perf_counter(), {}
for encoder, norm in RUNS:
    try:
        r = run_fronts(encoder, norm, df, DATA, FEAT, RES, token=os.environ["HF_TOKEN"])
        status[f"{encoder} | {norm}"] = "skipped (already saved)" if r is None else "done"
    except Exception as e:
        status[f"{encoder} | {norm}"] = f"FAILED: {type(e).__name__}: {str(e)[:300]}"
        print(status[f"{encoder} | {norm}"])
print(f"\n===== encoders finished in {(time.perf_counter() - t0) / 60:.0f} min =====")
for k, v in status.items():
    print(f"  {k:<32} {v}")

skip cradio-v4-h__caffe__tile512__na4: front results already on Drive
skip dinov3-l-photo__caffe__tile512__na4: front results already on Drive

===== dinov3-l-sat__caffe__tile512__na4 =====


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('train', 'val'): 559 of 681 images
  50/559  0.4 min elapsed, 0.46 s/image, ETA 3.9 min
  100/559  0.6 min elapsed, 0.39 s/image, ETA 3.0 min
  150/559  0.8 min elapsed, 0.32 s/image, ETA 2.2 min
  200/559  1.2 min elapsed, 0.37 s/image, ETA 2.2 min
  250/559  1.8 min elapsed, 0.44 s/image, ETA 2.2 min
  300/559  2.4 min elapsed, 0.48 s/image, ETA 2.1 min
  350/559  2.9 min elapsed, 0.50 s/image, ETA 1.8 min
  400/559  3.2 min elapsed, 0.48 s/image, ETA 1.3 min
  450/559  3.4 min elapsed, 0.45 s/image, ETA 0.8 min
  500/559  3.5 min elapsed, 0.42 s/image, ETA 0.4 min
  550/559  3.8 min elapsed, 0.42 s/image, ETA 0.1 min
  559/559  3.9 min elapsed, 0.42 s/image, ETA 0.0 min
in memory: train X (2148306, 1024) (4.4 GB fp16), 121 val images
MDE 1852 m over 121 SI images (6 without a predicted front) | extraction 4.0 min, total 6.3 min
skip saratrx-v1__unit__tile512__na4: front results already on Drive
skip terramind-v1-base-s1__

In [ ]:
# --- cell: U-NET, 4 classes, from scratch -> fronts -> MDE (not timed here; 48k training took 19.9 min in 05) ---
import pandas as pd
from sartransfer.baseline import load_split_images, train_unet
from sartransfer.runs import unet_fronts

ITERS = 48000                                # fixed before seeing results, as in 05
URUN = f"unet-scratch-{ITERS // 1000}k__caffe__tile256__na4"
if (RES / f"fronts_mde__{URUN}.csv").exists():
    print("skip", URUN, "-- already on Drive")
else:
    tr = load_split_images(df, DATA, "train", norm="caffe", with_na=True)
    va = load_split_images(df, DATA, "val", norm="caffe", with_na=True)
    assert len(tr) == 438 and len(va) == 121, "expected the real CaFFe train/SI split"
    unet, ulog = train_unet(tr, va, iters=ITERS, batch=16, crop=256, base=32,
                            log_every=12000, n_classes=4)   # SI check is info only
    recs, mde, zones = unet_fronts(unet, va, DATA, URUN)
    recs.to_csv(RES / f"fronts_images__{URUN}.csv", index=False)
    zones.to_csv(RES / f"zones4_pixel__{URUN}.csv", index=False)
    ulog.to_csv(RES / f"unet_log__{URUN}.csv", index=False)
    torch.save(unet.state_dict(), RES / f"unet__{URUN}.pt")
    mde.to_csv(RES / f"fronts_mde__{URUN}.csv", index=False)
    a = mde[mde.grouping == "all"].iloc[0]
    print(f"U-Net MDE {a.MDE_m:.0f} m over {a.images} SI images ({a.no_front} without a front)")
    del tr, va

skip unet-scratch-48k__caffe__tile256__na4 -- already on Drive


In [6]:
# --- cell: RESULTS -- MDE on SI for every model ---
import glob
pd.set_option("display.width", 200, "display.precision", 3)
mde = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RES / "fronts_mde__*.csv")))])
z4 = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RES / "zones4_pixel__*.csv")))])

overall = mde[mde.grouping == "all"].set_index("run")[["MDE_m", "images", "no_front", "n_dist"]]
iou = z4[(z4.grouping == "band") & (z4.group == "all")].set_index("run")[
    ["IoU_na", "IoU_stone", "IoU_glacier", "IoU_ocean", "mIoU"]]
table = overall.join(iou).sort_values("MDE_m")
assert (table["images"] == 121).all(), "every model must be scored on all 121 SI images"
print("=== MDE (m, lower is better) + 4-class pixel IoU, validation glacier SI ===")
print(table.round(3).to_string())
print("\n=== MDE per sensor @ resolution (m) ===")
print(mde[mde.grouping == "sensor_res"].pivot_table(index="run", columns="group", values="MDE_m").round(0))

=== MDE (m, lower is better) + 4-class pixel IoU, validation glacier SI ===
                                            MDE_m  images  no_front   n_dist  IoU_na  IoU_stone  IoU_glacier  IoU_ocean  mIoU
run                                                                                                                          
unet-scratch-48k__caffe__tile256__na4      1731.5     121         0  1001038     1.0        0.5          0.7        0.6   0.7
cradio-v4-h__caffe__tile512__na4           1816.3     121         5   662193     0.9        0.5          0.7        0.7   0.7
dinov3-l-sat__caffe__tile512__na4          1852.2     121         6   630628     0.9        0.4          0.6        0.6   0.7
dinov3-l-photo__caffe__tile512__na4        2018.4     121        10   542358     0.8        0.5          0.7        0.6   0.6
terramind-v1-base-s1__caffe__tile512__na4  2563.3     121         5   738953     0.7        0.4          0.4        0.3   0.4
saratrx-v1__unit__tile512__na4            

## Reading it

- **MDE** is in metres, lower is better. `no_front` = SI images where the model
  drew no front at all (excluded from MDE, as in the official code) -- a model
  with few distances but many `no_front` is not better.
- **4-class mIoU here includes NA and is pixel level**, the official zone
  metric's definition (the paper averages per image; this pools pixels).
  The IoU columns in the saved output above have one decimal (the cell printed with precision 1 then; it now prints three); docs/results.md (Arm 1:
  fronts on SI) has two decimals, except for the satellite DINOv3 probe.
- Frozen probes vs U-Net: the probes' front comes from a 16 px grid, the U-Net's
  from pixels. If the probes' MDE is far worse despite better zone mIoU, that is
  the grid limit -- the case for AnyUp, tried in 08.